In [1]:
from scapy.all import rdpcap, TCP, UDP, IP
from collections import defaultdict
import numpy as np
import pandas as pd

In [2]:
def get_payload_len(pkt):
    if TCP in pkt:
        return max(0, len(pkt[IP]) - pkt[IP].ihl * 4 - pkt[TCP].dataofs * 4)
    elif UDP in pkt:
        return max(0, len(pkt[IP]) - pkt[IP].ihl * 4 - 8)
    return 0


def get_hdr_len(pkt):
    if TCP in pkt:
        return pkt[IP].ihl * 4 + pkt[TCP].dataofs * 4
    return pkt[IP].ihl * 4 + 8


def iat_stats(times):
    if len(times) < 2:
        return 0, 0, 0, 0, 0
    iats = [(times[i + 1] - times[i]) * 1_000_000 for i in range(len(times) - 1)]
    return sum(iats), np.mean(iats), np.std(iats), max(iats), min(iats)


def get_packets(packets):
    flows = defaultdict(list)
    for pkt in packets:
        if IP in pkt and (TCP in pkt or UDP in pkt):
            ip = pkt[IP]
            proto = 'TCP' if TCP in pkt else 'UDP'
            if proto == 'TCP':
                sport, dport = pkt[TCP].sport, pkt[TCP].dport
            else:
                sport, dport = pkt[UDP].sport, pkt[UDP].dport
            key = (ip.src, sport, ip.dst, dport, proto)
            flows[key].append(pkt)
    return flows


def payload():
    sum_payload = sum(fwd_payload) if fwd_payload else 0
    max_payload = max(fwd_payload)
    min_payload = min(fwd_payload) if fwd_payload else 0
    mean_payload = np.mean(fwd_payload) if fwd_payload else 0
    std_payload = np.std(fwd_payload) if fwd_payload else 0
    return sum_payload, max_payload, min_payload, mean_payload, std_payload

In [3]:
THRESHOLD = 5_000_000  # 5 секунд в микросекундах (activityTimeout в CICFlowMeter)
packets = rdpcap('set0.pcap')
flows = get_packets(packets)


In [4]:
results = []

for key, pkts in flows.items():
    src_ip, sport, dst_ip, dport, proto = key
    
    fwd_pkts = [p for p in pkts if p[IP].src == src_ip and p[IP].dst == dst_ip]

    fwd_timestamps = [float(p.time) for p in fwd_pkts]
    fwd_iat_total, fwd_iat_mean, fwd_iat_std, fwd_iat_max, fwd_iat_min = iat_stats(fwd_timestamps)

    fwd_hdr = [get_hdr_len(p) for p in fwd_pkts]
    min_seg_size_forward = min(fwd_hdr) if fwd_hdr else 0
    fwd_hdr_sum = sum(fwd_hdr)

    fwd_payload = [get_payload_len(p) for p in fwd_pkts]

    sum_payload, max_payload, min_payload, mean_payload, std_payload = payload()

    bwd_pkts = [p for p in pkts if p[IP].src == dst_ip and p[IP].dst == src_ip]
    bwd_timestamps = [float(p.time) for p in bwd_pkts]
    bwd_payload = [get_payload_len(p) for p in bwd_pkts]
    bwd_hdr = [get_hdr_len(p) for p in bwd_pkts]
    bwd_iat_total, bwd_iat_mean, bwd_iat_std, bwd_iat_max, bwd_iat_min = iat_stats(bwd_timestamps)

    sum_bwd = sum(bwd_payload)
    max_bwd = max(bwd_payload) if bwd_payload else 0,
    min_bwd = min(bwd_payload) if bwd_payload else 0,
    mean_bwd = np.mean(bwd_payload) if bwd_payload else 0

    bwd_pkts = [p for p in pkts if p[IP].src == dst_ip and p[IP].dst == src_ip]
    bwd_timestamps = [float(p.time) for p in bwd_pkts]
    bwd_payload = [get_payload_len(p) for p in bwd_pkts]
    bwd_hdr = [get_hdr_len(p) for p in bwd_pkts]
    bwd_iat_total, bwd_iat_mean, bwd_iat_std, bwd_iat_max, bwd_iat_min = iat_stats(bwd_timestamps)

    timestamps = [float(p.time) for p in pkts]

    all_payload = fwd_payload + bwd_payload

    flow_iat_total, flow_iat_mean, flow_iat_std, flow_iat_max, flow_iat_min = iat_stats(timestamps)

    duration = (timestamps[-1] - timestamps[0]) * 1_000_000 if len(timestamps) > 1 else 0
    duration_s = duration / 1_000_000

    fwd_psh = sum(1 for p in fwd_pkts if TCP in p and p[TCP].flags & 0x08)
    bwd_psh = sum(1 for p in bwd_pkts if TCP in p and p[TCP].flags & 0x08)
    fwd_urg = sum(1 for p in fwd_pkts if TCP in p and p[TCP].flags & 0x20)
    bwd_urg = sum(1 for p in bwd_pkts if TCP in p and p[TCP].flags & 0x20)

    fin_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x01)
    syn_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x02)
    rst_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x04)
    psh_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x08)
    ack_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x10)
    urg_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x20)
    cwe_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x80)
    ece_count = sum(1 for p in pkts if TCP in p and p[TCP].flags & 0x40)

    init_win_fwd = fwd_pkts[0][TCP].window if fwd_pkts and TCP in fwd_pkts[0] else 0
    init_win_bwd = bwd_pkts[0][TCP].window if bwd_pkts and TCP in bwd_pkts[0] else 0

    act_data_pkt_fwd = sum(1 for p in fwd_pkts if get_payload_len(p) >= 1)


    # --- BULK ---
    def calc_bulk(packets_list, other_last_bulk_ts, own_last_bulk_ts):
        state_count = 0
        pkt_count = 0
        size_total = 0
        duration = 0
        pkt_count_helper = 0
        start_helper = 0
        size_helper = 0
        last_bulk_ts = own_last_bulk_ts

        for p in packets_list:
            ts = float(p.time) * 1_000_000
            size = get_payload_len(p)
            if other_last_bulk_ts > start_helper:
                start_helper = 0
            if size <= 0:
                continue
            if start_helper == 0:
                start_helper = ts
                pkt_count_helper = 1
                size_helper = size
                last_bulk_ts = ts
            else:
                if (ts - last_bulk_ts) / 1_000_000 > 1.0:
                    start_helper = ts
                    last_bulk_ts = ts
                    pkt_count_helper = 1
                    size_helper = size
                else:
                    pkt_count_helper += 1
                    size_helper += size
                    if pkt_count_helper == 4:
                        state_count += 1
                        pkt_count += pkt_count_helper
                        size_total += size_helper
                        duration += ts - start_helper
                    elif pkt_count_helper > 4:
                        pkt_count += 1
                        size_total += size
                        duration += ts - last_bulk_ts
                    last_bulk_ts = ts
        return state_count, pkt_count, size_total, duration


    f_bulk_state, f_bulk_pkts, f_bulk_size, f_bulk_dur = calc_bulk(fwd_pkts, 0, 0)
    b_bulk_state, b_bulk_pkts, b_bulk_size, b_bulk_dur = calc_bulk(bwd_pkts, 0, 0)


    def safe_div(a, b):
        return a / b if b != 0 else 0


    f_avg_bytes_bulk = safe_div(f_bulk_size, f_bulk_state)
    f_avg_pkts_bulk = safe_div(f_bulk_pkts, f_bulk_state)
    f_avg_bulk_rate = safe_div(f_bulk_size, f_bulk_dur / 1_000_000)
    b_avg_bytes_bulk = safe_div(b_bulk_size, b_bulk_state)
    b_avg_pkts_bulk = safe_div(b_bulk_pkts, b_bulk_state)
    b_avg_bulk_rate = safe_div(b_bulk_size, b_bulk_dur / 1_000_000)

    # --- SUBFLOW ---
    sf_count = 0
    sf_last_ts = -1
    for p in pkts:
        ts = float(p.time) * 1_000_000
        if sf_last_ts == -1:
            sf_last_ts = ts
        if (ts - sf_last_ts) / 1_000_000 > 1.0:
            sf_count += 1
        sf_last_ts = ts

    sf_fpackets = safe_div(len(fwd_pkts), sf_count) if sf_count > 0 else 0
    sf_fbytes = safe_div(sum_payload, sf_count) if sf_count > 0 else 0
    sf_bpackets = safe_div(len(bwd_pkts), sf_count) if sf_count > 0 else 0
    sf_bbytes = safe_div(sum(bwd_payload), sf_count) if sf_count > 0 else 0

    # --- ACTIVE/IDLE ---
    flow_active = []
    flow_idle = []
    start_active = timestamps[0]
    end_active = timestamps[0]
    for ts in timestamps[1:]:
        if (ts - end_active) * 1_000_000 > THRESHOLD:
            if (end_active - start_active) > 0:
                flow_active.append((end_active - start_active) * 1_000_000)
            flow_idle.append((ts - end_active) * 1_000_000)
            start_active = ts
            end_active = ts
        else:
            end_active = ts
    if (end_active - start_active) > 0:
        flow_active.append((end_active - start_active) * 1_000_000)

    total_bytes = sum(all_payload)
    total_pkts = len(pkts)

    flow = {

        'Flow ID': f"{src_ip}-{sport}-{dst_ip}-{dport}-{proto}",
        'Destination Port': dport,
        'Flow Duration': duration,
        'Total Fwd Packets': len(fwd_pkts),
        'Total Backward Packets': len(bwd_pkts),
        'Total Length of Fwd Packets': sum_payload,
        'Total Length of Bwd Packets': sum_bwd,
        'Fwd Packet Length Max': max_payload,
        'Fwd Packet Length Min': min_payload,
        'Fwd Packet Length Mean': mean_payload,
        'Fwd Packet Length Std': std_payload,
        'Bwd Packet Length Max': max_bwd,
        'Bwd Packet Length Min': min_bwd,
        'Bwd Packet Length Mean': mean_bwd,
        'Bwd Packet Length Std': np.std(bwd_payload) if bwd_payload else 0,
        'Flow Bytes/s': safe_div(total_bytes, duration_s),
        'Flow Packets/s': safe_div(total_pkts, duration_s),
        'Flow IAT Mean': flow_iat_mean,
        'Flow IAT Std': flow_iat_std,
        'Flow IAT Max': flow_iat_max,
        'Flow IAT Min': flow_iat_min,
        'Fwd IAT Total': fwd_iat_total,
        'Fwd IAT Mean': fwd_iat_mean,
        'Fwd IAT Std': fwd_iat_std,
        'Fwd IAT Max': fwd_iat_max,
        'Fwd IAT Min': fwd_iat_min,
        'Bwd IAT Total': bwd_iat_total,
        'Bwd IAT Mean': bwd_iat_mean,
        'Bwd IAT Std': bwd_iat_std,
        'Bwd IAT Max': bwd_iat_max,
        'Bwd IAT Min': bwd_iat_min,
        'Fwd PSH Flags': fwd_psh,
        'Bwd PSH Flags': bwd_psh,
        'Fwd URG Flags': fwd_urg,
        'Bwd URG Flags': bwd_urg,
        'Fwd Header Length': fwd_hdr_sum,
        'Bwd Header Length': sum(bwd_hdr),
        'Fwd Packets/s': safe_div(len(fwd_pkts), duration_s),
        'Bwd Packets/s': safe_div(len(bwd_pkts), duration_s),
        'Min Packet Length': min(all_payload) if all_payload else 0,
        'Max Packet Length': max(all_payload) if all_payload else 0,
        'Packet Length Mean': np.mean(all_payload) if all_payload else 0,
        'Packet Length Std': np.std(all_payload) if all_payload else 0,
        'Packet Length Variance': np.var(all_payload) if all_payload else 0,
        'FIN Flag Count': fin_count,
        'SYN Flag Count': syn_count,
        'RST Flag Count': rst_count,
        'PSH Flag Count': psh_count,
        'ACK Flag Count': ack_count,
        'URG Flag Count': urg_count,
        'CWE Flag Count': cwe_count,
        'ECE Flag Count': ece_count,
        'Down/Up Ratio': safe_div(len(bwd_pkts), len(fwd_pkts)),
        'Average Packet Size': safe_div(total_bytes, total_pkts),
        'Avg Fwd Segment Size': mean_payload if fwd_payload else 0,
        'Avg Bwd Segment Size': np.mean(bwd_payload) if bwd_payload else 0,
        'Fwd Avg Bytes/Bulk': f_avg_bytes_bulk,
        'Fwd Avg Packets/Bulk': f_avg_pkts_bulk,
        'Fwd Avg Bulk Rate': f_avg_bulk_rate,
        'Bwd Avg Bytes/Bulk': b_avg_bytes_bulk,
        'Bwd Avg Packets/Bulk': b_avg_pkts_bulk,
        'Bwd Avg Bulk Rate': b_avg_bulk_rate,
        'Subflow Fwd Packets': sf_fpackets,
        'Subflow Fwd Bytes': sf_fbytes,
        'Subflow Bwd Packets': sf_bpackets,
        'Subflow Bwd Bytes': sf_bbytes,
        'Init_Win_bytes_forward': init_win_fwd,
        'Init_Win_bytes_backward': init_win_bwd,
        'act_data_pkt_fwd': act_data_pkt_fwd,
        'min_seg_size_forward': min_seg_size_forward,
        'Active Mean': np.mean(flow_active) if flow_active else 0,
        'Active Std': np.std(flow_active) if flow_active else 0,
        'Active Max': max(flow_active) if flow_active else 0,
        'Active Min': min(flow_active) if flow_active else 0,
        'Idle Mean': np.mean(flow_idle) if flow_idle else 0,
        'Idle Std': np.std(flow_idle) if flow_idle else 0,
        'Idle Max': max(flow_idle) if flow_idle else 0,
        'Idle Min': min(flow_idle) if flow_idle else 0,
    }
    results.append(flow)

import pandas as pd

pcap_df = pd.DataFrame(results)
pcap_df.columns = pcap_df.columns.str.strip()
display(pcap_df)

,Flow ID,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,192.168.1.239-5353-224.0.0.251-5353-UDP,5353,0.000000,1,0,652,0,652,652,652.00,...,1,28,0.000000,0.0,0.000000,0.000000,0,0,0,0
1,192.168.1.216-5353-224.0.0.251-5353-UDP,5353,0.000000,1,0,658,0,658,658,658.00,...,1,28,0.000000,0.0,0.000000,0.000000,0,0,0,0
2,17.248.202.64-443-192.168.1.244-55213-TCP,55213,12896.060944,4,0,65,0,39,0,16.25,...,2,52,12896.060944,0.0,12896.060944,12896.060944,0,0,0,0
3,192.168.1.244-55213-17.248.202.64-443-TCP,443,1644.849777,4,0,63,0,39,0,15.75,...,2,52,1644.849777,0.0,1644.849777,1644.849777,0,0,0,0
4,192.168.1.244-49484-142.251.41.3-443-TCP,443,0.000000,1,0,0,0,0,0,0.00,...,0,40,0.000000,0.0,0.000000,0.000000,0,0,0,0
